In [1]:
# Instalação do PySpar
!pip install pyspark -q

In [2]:
#biblioteca com confirmação
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("OtimizacaoJoinVideosComments")
.getOrCreate() )

print("Spark iniciado com sucesso!")

Spark iniciado com sucesso!


In [14]:
# leitura dos dados
videos_file = "/content/videos-preparados.snappy (1).parquet"
comments_file = "/content/videos-comments-tratados.snappy (1).parquet"

df_video = spark.read.parquet(videos_file)
df_comments = spark.read.parquet(comments_file)

print("Schema df_video:")
df_video.printSchema()

print("Schema df_comments:")
df_comments.printSchema()

Schema df_video:
root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)

Schema df_comments:
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string (nulla

In [20]:
#importação dos dados pelo drive
from google.colab import drive
drive.mount('/content/drive')

print('Google Drive montado com sucesso!')

Mounted at /content/drive
Google Drive montado com sucesso!


Após montar o Google Drive, seus arquivos estarão disponíveis em `/content/drive/MyDrive/`. Você precisará ajustar os caminhos das variáveis `videos_file` e `comments_file` no seu notebook para apontar para a localização correta dos arquivos no seu Drive.

# Nova seção

In [21]:
# print das primeiras linhas
print("Primeiras linhas de df_video:")
df_video.show(5, truncate=False)

print("Primeiras linhas de df_comments:")
df_comments.show(5, truncate=False)

Primeiras linhas de df_video:
+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+--------------------------------------+
|Title                                                                                                |Video ID   |Published At|Keyword|Likes |Comments|Views   |Interaction|Year|Month|Keyword Index|Features PCA         |Features Normal                                                 |Features                              |
+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+-----------------------------------

In [17]:
# Realização do Join
joined_df_api = df_video.join(df_comments, on="Video ID", how="inner")

print("Primeiras linhas do DataFrame resultante do join (DataFrame API):")
joined_df_api.show(5, truncate=False)

Primeiras linhas do DataFrame resultante do join (DataFrame API):
+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+-------------------------------------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------

In [22]:
#tabela
df_video.createOrReplaceTempView("tb_videos")
df_comments.createOrReplaceTempView("tb_comments")

print("Tabelas temporárias criadas com sucesso.")

Tabelas temporárias criadas com sucesso.


In [23]:
#SQL
join_video_comments = spark.sql("""SELECT v.*, c.*
FROM tb_videos v
INNER JOIN tb_comments c
ON v.`Video ID` = c.`Video ID` """)

join_video_comments.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+-------------------------------------------------------------+---------------------------------+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+
|Title                             

In [24]:
#Leitura
df_video_rep = spark.read.parquet(videos_file).repartition(4, "Video ID")
df_comments_rep = spark.read.parquet(comments_file).repartition(4, "Video ID")

#Tabela
df_video_rep.createOrReplaceTempView("tb_videos_rep")
df_comments_rep.createOrReplaceTempView("tb_comments_rep")

#Join sql
join_video_comments_rep = spark.sql(""" SELECT v.*, c.*
FROM tb_videos_rep v
INNER JOIN tb_comments_rep c
ON v.`Video ID` = c.`Video ID` """)

# 4) Visualização
join_video_comments_rep.show(5, truncate=False)

+------------------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------------------------------------------------+-----------------------------------+-----------+------------------------------+------------+-------+-----+--------+-------+-----------+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------

In [25]:

df_video_coa = spark.read.parquet(videos_file).coalesce(2)
df_comments_coa = spark.read.parquet(comments_file).coalesce(2)


df_video_coa.createOrReplaceTempView("tb_videos_coa")
df_comments_coa.createOrReplaceTempView("tb_comments_coa")

join_video_comments_coa = spark.sql(""" SELECT v.*, c.*
FROM tb_videos_coa v
INNER JOIN tb_comments_coa c
ON v.`Video ID` = c.`Video ID` """)

#Visualização
join_video_comments_coa.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+-------------------------------------------------------------+---------------------------------+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+
|Title                             

In [26]:
#Visualização
print("Explain com repartition:")
join_video_comments_rep.explain(True)

print("\n" + "="*100 + "\n")

print("Explain com coalesce:")
join_video_comments_coa.explain(True)

Explain com repartition:
== Parsed Logical Plan ==
'Project [v.*, c.*]
+- 'Join Inner, ('v.Video ID = 'c.Video ID)
   :- 'SubqueryAlias v
   :  +- 'UnresolvedRelation [tb_videos_rep], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [tb_comments_rep], [], false

== Analyzed Logical Plan ==
Title: string, Video ID: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: int, Month: int, Keyword Index: double, Features PCA: vector, Features Normal: vector, Features: vector, Video ID: string, Title: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: string, Comment: string, Sentiment: int, ... 1 more fields
Project [Title#263, Video ID#264, Published At#265, Keyword#266, Likes#267, Comments#268, Views#269, Interaction#270, Year#271, Month#272, Keyword Index#273, Features PCA#274, Features Normal#275, Features#276, Video ID#278, Title#279, Published At#280, Keyword#281,

In [27]:
#importação
from pyspark.sql.functions import col

# verificação daas colunas
video_cols = ["Video ID", "Title", "Keyword", "Views", "Likes", "Published At"]

df_video_otim = ( spark.read.parquet(videos_file)
.select(*[col(c) for c in video_cols if c in df_video.columns])
.filter(col("Video ID").isNotNull())
.repartition(4, "Video ID") )

comments_cols = ["Video ID", "Comment", "Sentiment"]

df_comments_otim = (spark.read.parquet(comments_file)
.select(*[col(c) for c in comments_cols if c in df_comments.columns])
.filter(col("Video ID").isNotNull())
.repartition(4, "Video ID") )

# tabela
df_video_otim.createOrReplaceTempView("tb_videos_otim")
df_comments_otim.createOrReplaceTempView("tb_comments_otim")

# Join
join_video_comments_otimizado = spark.sql(""" SELECT
v.`Video ID`,
  v.Title,
    v.Keyword,
      v.Views,
        v.Likes,
      v.`Published At`,
    c.Comment,
  c.Sentiment
FROM tb_videos_otim v INNER JOIN tb_comments_otim c ON v.`Video ID` = c.`Video ID`
""")

join_video_comments_otimizado.explain(True)

# Resultado
join_video_comments_otimizado.show(10, truncate=False)

== Parsed Logical Plan ==
'Project ['v.Video ID, 'v.Title, 'v.Keyword, 'v.Views, 'v.Likes, 'v.Published At, 'c.Comment, 'c.Sentiment]
+- 'Join Inner, ('v.Video ID = 'c.Video ID)
   :- 'SubqueryAlias v
   :  +- 'UnresolvedRelation [tb_videos_otim], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [tb_comments_otim], [], false

== Analyzed Logical Plan ==
Video ID: string, Title: string, Keyword: string, Views: int, Likes: int, Published At: date, Comment: string, Sentiment: int
Project [Video ID#476, Title#475, Keyword#478, Views#481, Likes#479, Published At#477, Comment#499, Sentiment#500]
+- Join Inner, (Video ID#476 = Video ID#490)
   :- SubqueryAlias v
   :  +- SubqueryAlias tb_videos_otim
   :     +- View (`tb_videos_otim`, [Video ID#476, Title#475, Keyword#478, Views#481, Likes#479, Published At#477])
   :        +- RepartitionByExpression [Video ID#476], 4
   :           +- Filter isnotnull(Video ID#476)
   :              +- Project [Video ID#476, Title#475, Keyword#

In [28]:
# Otimização
df_video_otim_2 = ( spark.read.parquet(videos_file)
.select(*[col(c) for c in video_cols if c in df_video.columns])
.filter(col("Video ID").isNotNull())
.filter(col("Views") >= 0)
.repartition(4, "Video ID") )

df_comments_otim_2 = ( spark.read.parquet(comments_file)
.select(*[col(c) for c in comments_cols if c in df_comments.columns])
.filter(col("Video ID").isNotNull())
.repartition(4, "Video ID") )

join_video_comments_otimizado_2 = ( df_video_otim_2.alias("v") .join(
df_comments_otim_2.alias("c"),
on="Video ID",
how="inner")
)

# Mostra o plano de execução detalhado
join_video_comments_otimizado_2.explain(True)

# Exibe algumas linhas do resultado
join_video_comments_otimizado_2.show(10, truncate=False)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [Video ID])
:- SubqueryAlias v
:  +- RepartitionByExpression [Video ID#529], 4
:     +- Filter (Views#534 >= 0)
:        +- Filter isnotnull(Video ID#529)
:           +- Project [Video ID#529, Title#528, Keyword#531, Views#534, Likes#532, Published At#530]
:              +- Relation [Title#528,Video ID#529,Published At#530,Keyword#531,Likes#532,Comments#533,Views#534,Interaction#535,Year#536,Month#537,Keyword Index#538,Features PCA#539,Features Normal#540,Features#541] parquet
+- SubqueryAlias c
   +- RepartitionByExpression [Video ID#543], 4
      +- Filter isnotnull(Video ID#543)
         +- Project [Video ID#543, Comment#552, Sentiment#553]
            +- Relation [Video ID#543,Title#544,Published At#545,Keyword#546,Likes#547,Comments#548,Views#549,Interaction#550,Year#551,Comment#552,Sentiment#553,Likes Comment#554] parquet

== Analyzed Logical Plan ==
Video ID: string, Title: string, Keyword: string, Views: int, Likes: int, Publishe

In [29]:
#Tratamento ou filtro
df_video_otim_2 = (spark.read.parquet(videos_file)
.select(*[col(c) for c in video_cols if c in df_video.columns])
.filter(col("Video ID").isNotNull())
.filter(col("Views") >= 0)
.repartition(4, "Video ID") )

df_comments_otim_2 = ( spark.read.parquet(comments_file)
.select(*[col(c) for c in comments_cols if c in df_comments.columns])
.filter(col("Video ID").isNotNull())
.repartition(4, "Video ID") )

#Join
join_video_comments_otimizado_2 = ( df_video_otim_2.alias("v") .join(
df_comments_otim_2.alias("c"),
on="Video ID",
how="inner")
)


join_video_comments_otimizado_2.explain(True)

#Resultado
join_video_comments_otimizado_2.show(10, truncate=False)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [Video ID])
:- SubqueryAlias v
:  +- RepartitionByExpression [Video ID#582], 4
:     +- Filter (Views#587 >= 0)
:        +- Filter isnotnull(Video ID#582)
:           +- Project [Video ID#582, Title#581, Keyword#584, Views#587, Likes#585, Published At#583]
:              +- Relation [Title#581,Video ID#582,Published At#583,Keyword#584,Likes#585,Comments#586,Views#587,Interaction#588,Year#589,Month#590,Keyword Index#591,Features PCA#592,Features Normal#593,Features#594] parquet
+- SubqueryAlias c
   +- RepartitionByExpression [Video ID#596], 4
      +- Filter isnotnull(Video ID#596)
         +- Project [Video ID#596, Comment#605, Sentiment#606]
            +- Relation [Video ID#596,Title#597,Published At#598,Keyword#599,Likes#600,Comments#601,Views#602,Interaction#603,Year#604,Comment#605,Sentiment#606,Likes Comment#607] parquet

== Analyzed Logical Plan ==
Video ID: string, Title: string, Keyword: string, Views: int, Likes: int, Publishe

In [30]:
#Saida e salvamento
output_path = "/content/join-videos-comments-otimizado"

join_video_comments_otimizado_2.write.mode("overwrite").parquet(output_path)

print(f"Arquivo salvo com sucesso em: {output_path}")

Arquivo salvo com sucesso em: /content/join-videos-comments-otimizado


In [31]:
#Resuktado do salvamenro
df_resultado = spark.read.parquet(output_path)

print("Schema do arquivo salvo:")
df_resultado.printSchema()

print("Amostra do resultado salvo:")
df_resultado.show(10, truncate=False)

Schema do arquivo salvo:
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)

Amostra do resultado salvo:
+-----------+--------------------------------------------------------------------------------------------------+-------+------+-----+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+
|Video ID

In [32]:
#Nome das colunas
df_video.columns
df_comments.columns

['Video ID',
 'Title',
 'Published At',
 'Keyword',
 'Likes',
 'Comments',
 'Views',
 'Interaction',
 'Year',
 'Comment',
 'Sentiment',
 'Likes Comment']